<a href="https://colab.research.google.com/github/Giraffe-Shin/trading/blob/main/copper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 구글 드라이브 마운트 예시 (필요할 때 실행하세요)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install fredapi
import pandas as pd
import yfinance as yf
from fredapi import Fred
from google.colab import userdata
import os
import time

try:
    FRED_API_KEY = userdata.get('FRED_API_KEY')
except Exception:
    FRED_API_KEY = os.getenv('FRED_API_KEY')

if FRED_API_KEY:
    fred = Fred(api_key=FRED_API_KEY)
else:
    print("Warning: FRED_API_KEY not found.")
    fred = None

def fetch_fred_data(ticker_id):
    if fred:
        print(f"Fetching FRED data for {ticker_id}...")
        try:
            df = fred.get_series(ticker_id)
            if df is not None:
                df = df.to_frame(name=ticker_id)
                df.index.name = 'Date'
                return df
            return pd.DataFrame()
        except Exception as e:
            print(f"Error fetching FRED data for {ticker_id}: {e}")
            return pd.DataFrame()
    return pd.DataFrame()

def load_data_from_source(source_type, ticker_id, **kwargs):
    if source_type == 'St. Louis FRED':
        return fetch_fred_data(ticker_id)
    elif source_type == 'Yahoo Finance':
        return yf.download(ticker_id, **kwargs)
    return pd.DataFrame()

print("Setup complete with time module imported.")

## 추가 FRED 데이터 불러오기 및 시각화

이제 사용자가 요청한 추가 FRED 티커들을 수요, 공급, 가격 범주로 나누어 불러오고 시각화합니다. 각 그래프에는 티커에 대한 요약 설명이 포함됩니다.

In [ ]:
import time

# 모든 카테고리별 FRED 티커 정의
fred_tickers = {
    'demand': {
        'WPU10260301': 'PPI: Secondary Smelting, Refining, Alloying of Copper',
        'WPU102502': 'PPI: Primary Smelting and Refining of Copper',
        'HOUST': 'Housing Starts: Total New Privately Owned Housing Units Started',
        'WPU1412011': 'PPI: Primary Nonferrous Battery Materials (incl. Lithium)'
    },
    'supply': {
        'IPG21223S': 'Industrial Production: Mining: Copper, Nickel, Lead, and Zinc Mining',
        'WPU10230103': 'PPI: Nonferrous Metals: Refined Copper',
        'CAPUTLG2122S': 'Capacity Utilization: Mining (except Oil and Gas)',
        'WPU101707': 'PPI: Copper and Copper Alloy Scrap'
    },
    'price': {
        'WPUSI019011': 'PPI: Metals and Metal Products: Copper Wire and Cable',
        'PCOPPUSDM': 'Global price of Copper (USD per Metric Ton, Monthly)',
        'PALUMUSDM': 'Global price of Aluminum (USD per Metric Ton, Monthly)'
    },
    'outlook': {
        'NASDAQNSCOPP': 'NASDAQ Copper Index (Market Outlook & Sentiment)'
    }
}

# 데이터프레임 초기화
fred_dataframes = {}

print("전체 FRED 데이터 다시 수집 중...")
for category, tickers in fred_tickers.items():
    for ticker_id, description in tickers.items():
        df = load_data_from_source('St. Louis FRED', ticker_id)
        if not df.empty:
            fred_dataframes[ticker_id] = {'data': df, 'description': description, 'category': category}
            print(f"  [성공] {category.upper()} - {ticker_id}")
        else:
            print(f"  [실패] {category.upper()} - {ticker_id}")
        time.sleep(1) # 간격 조정

print(f"\n수집 완료: 총 {len(fred_dataframes)}개 지표 확보")

In [ ]:
# 공급(supply) 카테고리 데이터만 추출하여 하나의 DataFrame으로 통합
supply_dfs = []

for ticker_id, info in fred_dataframes.items():
    if info['category'] == 'supply':
        df_temp = info['data'].copy()
        supply_dfs.append(df_temp)

if supply_dfs:
    # 날짜(Date) 기준으로 모든 공급 데이터 병합
    df_supply = pd.concat(supply_dfs, axis=1)
    df_supply.sort_index(inplace=True)

    print("공급 관련 데이터프레임 통합 완료 (df_supply)")
    display(df_supply.tail())
else:
    print("통합할 공급 관련 데이터가 없습니다.")

In [ ]:
import pandas as pd
import os

# 1. 저장 경로 설정
target_dir = '/content/drive/MyDrive/Colab Notebooks/electricity_analysis'
if not os.path.exists(target_dir):
    os.makedirs(target_dir)

file_path = os.path.join(target_dir, 'fred_total_data.csv')

# 2. 모든 카테고리의 데이터를 통합
if 'fred_dataframes' in globals() and fred_dataframes:
    all_dfs = []
    print(f"통합 대상:")
    for ticker, info in fred_dataframes.items():
        if not info['data'].empty:
            all_dfs.append(info['data'])
            print(f"  - {ticker} ({info['category']})")

    # Outer Join으로 병합
    df_new = pd.concat(all_dfs, axis=1)
    df_new.index.name = 'Date'
    df_new.sort_index(inplace=True)

    # 3. 구글 드라이브 파일 저장 (덮어쓰기 또는 업데이트)
    df_new.to_csv(file_path)
    print(f"\n[완료] 모든 지표가 포함된 파일이 저장되었습니다: {file_path}")
    print(f"최종 통합 결과: {df_new.shape[1]}개 열, {len(df_new)}개 행")
    display(df_new.tail())
else:
    print("통합할 수 있는 데이터가 없습니다.")

In [ ]:
import pandas as pd

# 통합된 데이터프레임의 구성 확인
print("=== df_new 데이터 구성 요약 ===")

# 카테고리별로 포함된 티커 분류
structure = {}
for ticker, info in fred_dataframes.items():
    cat = info['category']
    if cat not in structure:
        structure[cat] = []
    structure[cat].append(f"{ticker} ({info['description'][:30]}...)")

for cat, items in structure.items():
    print(f"\n[{cat.upper()}] 카테고리 ({len(items)}개):")
    for item in items:
        print(f"  - {item}")

print(f"\n총 통합 열(Column) 수: {df_new.shape[1]}개")
print(f"데이터 기간: {df_new.index.min().date()} ~ {df_new.index.max().date()}")

### 💾 데이터 로컬 저장 및 관리 방법

매번 API를 호출하는 것은 속도가 느리고 호출 제한에 걸릴 수 있습니다. 가장 일반적인 방법은 다음과 같습니다:

1.  **Colab 로컬 세션 저장**: `df.to_csv()`를 사용하여 현재 세션의 파일 시스템에 저장합니다 (세션 종료 시 삭제됨).
2.  **Google Drive 마운트**: 드라이브를 연결하여 `/content/drive/My Drive/` 경로에 영구적으로 저장합니다.

먼저 현재까지 수집된 데이터를 CSV 파일로 저장해 보겠습니다.

In [ ]:
import os

# 파일 저장 경로 설정
file_path = 'fred_total_data.csv'

# df_fred_total이 존재하는지 확인
if 'df_fred_total' in globals():
    df_fred_total.to_csv(file_path)
    print(f"성공: 데이터를 '{file_path}'로 저장했습니다.")
    print(f"파일 크기: {os.path.getsize(file_path) / 1024:.2f} KB")
else:
    print("저장할 데이터프레임(df_fred_total)이 없습니다. 먼저 데이터 통합 셀을 완료해주세요.")

위의 파일은 왼쪽 사이드바의 **폴더 아이콘(Files)**을 클릭하면 확인할 수 있습니다.

만약 **Google Drive**에 영구 저장하고 싶으시다면, 아래 코드를 실행하여 마운트할 수 있습니다. 마운트 후에는 경로를 `/content/drive/MyDrive/filename.csv`로 지정하면 됩니다.

In [ ]:
import pandas as pd

# 1. 데이터 불러오기
file_path = '/content/drive/MyDrive/Colab Notebooks/electricity_analysis/fred_total_data.csv'
copper = pd.read_csv(file_path, index_col='Date', parse_dates=True)

# 2. 직관적인 컬럼명 매핑 정의
rename_map = {
    'WPU10260301': 'PPI_Secondary_Copper',      # 구리 2차 제련 물가
    'WPU102502': 'PPI_Primary_Copper',          # 구리 1차 제련 물가
    'HOUST': 'Housing_Starts',                  # 주택 착공 건수 (수요 지표)
    'IPG21223S': 'Production_Copper_Zinc_Mining', # 구리/아연 광업 생산
    'WPU10230103': 'PPI_Refined_Copper',        # 정제 구리 물가
    'CAPUTLG2122S': 'Mining_Capacity_Usage',    # 광업 가동률
    'WPU101707': 'PPI_Copper_Scrap',            # 구리 스크랩 물가
    'WPUSI019011': 'PPI_Copper_Wire_Cable',     # 구리 전선/케이블 물가
    'PCOPPUSDM': 'Copper_Price_Global',         # 글로벌 구리 가격
    'PALUMUSDM': 'Aluminum_Price_Global',       # 글로벌 알루미늄 가격
    'NASDAQNSCOPP': 'NASDAQ_Copper_Index'       # 나스닥 구리 지수 (전망)
}

# 3. 컬럼명 변경
copper.rename(columns=rename_map, inplace=True)

# 4. 변경된 결과 저장 및 확인
copper.to_csv(file_path)
print("=== 컬럼명 변경 및 저장 완료 ===")
copper.info()
display(copper.tail())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 원본 데이터의 결측치 시각화 (Missing Value Heatmap)
plt.figure(figsize=(12, 6))
sns.heatmap(pd.read_csv(file_path, index_col='Date').isnull(), cbar=False, cmap='viridis')
plt.title('Missing Value Heatmap (Original Data)')
plt.show()

# 2. 데이터 신뢰도가 높은 2010년 이후 데이터로 필터링
# (나스닥 지수 등 최신 지표의 결측 구간이 너무 길기 때문)
copper_final = copper[copper.index >= '2010-01-01']

# 3. 최종 데이터 요약 및 저장
print("\n=== [확정] 분석용 최종 데이터셋 (2010년 ~ 현재) ===")
print(f"데이터 크기: {copper_final.shape}")
print(f"최종 데이터 시작일: {copper_final.index.min().date()}")
print(f"최종 데이터 종료일: {copper_final.index.max().date()}")

# 보정된 최종 데이터를 변수에 할당
copper = copper_final
copper.tail()

In [ ]:
import matplotlib.pyplot as plt

# 원본 데이터(결측치 포함) 다시 읽기
raw_data = pd.read_csv(file_path, index_col='Date', parse_dates=True)
raw_data.rename(columns=rename_map, inplace=True)

# 특정 지표(예: PPI_Primary_Copper)를 선택해 보간 전후 비교
target_col = 'PPI_Primary_Copper'

plt.figure(figsize=(15, 5))
# 1. 보간된 데이터 (선)
plt.plot(copper.index, copper[target_col], label='Interpolated (Daily)', color='skyblue', alpha=0.8)
# 2. 원본 데이터 (실제 값만 점으로 표시)
original_points = raw_data[raw_data.index >= '2010-01-01'][target_col].dropna()
plt.scatter(original_points.index, original_points.values, color='red', s=10, label='Actual Data Points', zorder=3)

plt.title(f'Data Integrity Check: {target_col} (2010-2024)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"빨간 점이 실제 데이터이며, 하늘색 선이 보간된 부분입니다.")
print(f"월간 데이터 사이를 직선으로 연결했기 때문에 급격한 왜곡 없이 흐름을 잘 따르고 있음을 알 수 있습니다.")

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import matplotlib.pyplot as plt

# 1. 시계열 예측을 위한 피처 엔지니어링 (Lag Features 추가)
copper_model_df = copper.copy()

# 타겟인 구리 가격의 과거 1일, 7일 전 값을 새로운 특성으로 추가
copper_model_df['Price_Lag1'] = copper_model_df['Copper_Price_Global'].shift(1)
copper_model_df['Price_Lag7'] = copper_model_df['Copper_Price_Global'].shift(7)

# 결측치 제거 (Shift로 인해 발생한 앞부분)
copper_model_df = copper_model_df.dropna()

# 2. 데이터 재분할
target_col = 'Copper_Price_Global'
X = copper_model_df.drop(columns=[target_col])
y = copper_model_df[target_col]

split_idx = int(len(copper_model_df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# 3. 모델 재학습
rf_improved = RandomForestRegressor(n_estimators=100, random_state=42)
rf_improved.fit(X_train, y_train)

# 4. 예측 및 평가
y_pred = rf_improved.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"=== 개선된 모델 평가 결과 (Lag Feature 추가) ===")
print(f"RMSE: {rmse:.2f}")
print(f"R2 Score: {r2:.4f}")

# 5. 실제값 vs 예측값 시각화
plt.figure(figsize=(15, 5))
plt.plot(y_test.index, y_test.values, label='Actual Price', color='black', alpha=0.7)
plt.plot(y_test.index, y_pred, label='Predicted Price (with Lag)', color='blue', linestyle='--')
plt.title('Improved Copper Price Prediction (Testing Data)')
plt.legend()
plt.show()

In [ ]:
import sys
# PyTorch Forecasting 및 관련 라이브러리 설치
!pip install pytorch-forecasting lightning


### 1. TFT 전용 데이터셋 준비
TFT 모델은 일반 회귀 모델과 달리 시간 인덱스(`time_idx`)와 그룹 식별자(`group`)가 필요합니다.
또한 데이터의 스케일 차이가 크므로 적절한 정규화 과정이 포함됩니다.

In [ ]:
import pandas as pd
import numpy as np
from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer

# TFT 학습을 위해 데이터 복사
tft_df = copper.copy()

# 1. 시간 인덱스(time_idx) 생성 (0부터 시작하는 정수)
tft_df['time_idx'] = (tft_df.index - tft_df.index.min()).days

# 2. 그룹 ID 추가 (단일 시계열이므로 모두 'copper'로 설정)
tft_df['group'] = 'copper'

# 3. 모델이 예측에 참고할 수 있도록 '월(Month)' 정보 추출
tft_df['month'] = tft_df.index.month.astype(str).astype('category')

# 데이터 확인
print(f"TFT 준비 완료된 데이터 크기: {tft_df.shape}")
display(tft_df.head())


In [ ]:
from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer

# 1. 설정값 정의
max_prediction_length = 30  # 30일 예측
max_encoder_length = 180    # 180일(6개월) 과거 데이터 참고
training_cutoff = tft_df["time_idx"].max() - max_prediction_length

# 2. 데이터셋 정의
training = TimeSeriesDataSet(
    tft_df[lambda x: x.time_idx <= training_cutoff],
    time_idx="time_idx",
    target="Copper_Price_Global",
    group_ids=["group"],
    min_encoder_length=max_encoder_length // 2,
    max_encoder_length=max_encoder_length,
    min_prediction_length=1,
    max_prediction_length=max_prediction_length,
    static_categoricals=["group"],
    time_varying_known_categoricals=["month"],
    time_varying_known_reals=["time_idx"],
    time_varying_unknown_reals=[
        "Copper_Price_Global",
        "PPI_Primary_Copper",
        "PPI_Secondary_Copper",
        "Housing_Starts",
        "PPI_Copper_Wire_Cable",
        "NASDAQ_Copper_Index"
    ],
    target_normalizer=GroupNormalizer(
        groups=["group"], transformation="softplus"
    ),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=True  # 날짜 간격 불일치 허용
)

# 3. 검증 데이터셋 생성
validation = TimeSeriesDataSet.from_dataset(training, tft_df, predict=True, stop_randomization=True)

# 4. 데이터로더 생성
batch_size = 64
train_dataloader = training.to_dataloader(train=True, batch_size=batch_size, num_workers=0)
val_dataloader = validation.to_dataloader(train=False, batch_size=batch_size * 10, num_workers=0)

print(f"학습용 샘플 수: {len(training)}")
print(f"검증용 샘플 수: {len(validation)}")

In [ ]:
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor
from pytorch_forecasting import TemporalFusionTransformer, QuantileLoss

# 1. 모델 인스턴스 생성
tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.03,
    hidden_size=16,  # 데이터셋 크기를 고려하여 작은 사이즈로 시작
    attention_head_size=4,
    dropout=0.1,
    hidden_continuous_size=8,
    loss=QuantileLoss(),  # 분위수 손실 함수 (예측 범위를 제공)
    log_interval=10,
    reduce_on_plateau_patience=4
)

print(f"모델 파라미터 수: {tft.size()/1e3:.1f}k")

# 2. 트레이너(Trainer) 설정
trainer = pl.Trainer(
    max_epochs=30,
    accelerator="cpu", # GPU 가용 시 "auto" 혹은 "gpu"
    enable_model_summary=True,
    gradient_clip_val=0.1,
    callbacks=[
        EarlyStopping(monitor="val_loss", min_delta=1e-4, patience=10, verbose=False, mode="min"),
        LearningRateMonitor(logging_interval="step")
    ]
)

# 3. 학습 시작
trainer.fit(
    tft,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader
)

In [ ]:
import matplotlib.pyplot as plt

# 1. 검증 데이터에 대한 예측 수행
raw_predictions = tft.predict(val_dataloader, mode="raw", return_x=True)

# 2. 예측 결과 시각화 (Best vs Actual)
plt.figure(figsize=(10, 5))
tft.plot_prediction(raw_predictions.x, raw_predictions.output, idx=0, add_loss_to_title=True)
plt.title("Copper Price Prediction: Actual vs Predicted (Validation Set)")
plt.show()

# 3. 변수 중요도(Variable Importance) 확인
interpretation = tft.interpret_output(raw_predictions.output, reduction="sum")
tft.plot_interpretation(interpretation)
plt.show()

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. 실제 값과 p50(중간값) 예측치 추출
actuals = torch.cat([y[0] for x, y in iter(val_dataloader)]).cpu().numpy().flatten()

# raw_predictions.output.prediction 구조: (batch, time, quantiles)
# QuantileLoss 기본값 사용 시 index 3이 median(p50)
preds_tensor = raw_predictions.output.prediction
predictions = preds_tensor[:, :, 3].cpu().numpy().flatten()

# 2. 성능 지표 계산
rmse = np.sqrt(mean_squared_error(actuals, predictions))
mae = mean_absolute_error(actuals, predictions)
r2 = r2_score(actuals, predictions)

# 3. 결과 출력 및 해석
print(f"=== Temporal Fusion Transformer (TFT) 모델 평가 결과 ===")
print(f"1. RMSE (Root Mean Squared Error): {rmse:.2f}")
print(f"   - 해석: 예측값이 실제 가격에서 평균적으로 약 {rmse:.2f}달러 정도 벗어나 있습니다.")
print(f"2. MAE (Mean Absolute Error): {mae:.2f}")
print(f"   - 해석: 절대적인 오차의 평균이며, 이상치에 덜 민감한 지표입니다.")
print(f"3. R2 Score (Coefficient of Determination): {r2:.4f}")
print(f"   - 해석: 1.0에 가까울수록 모델이 가격 변동 패턴을 완벽하게 설명함을 의미합니다.")

# 4. 오차 분포(Residuals) 시각화
residuals = actuals - predictions
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(predictions, actuals, alpha=0.5)
plt.plot([actuals.min(), actuals.max()], [actuals.min(), actuals.max()], 'r--', lw=2)
plt.xlabel('Predicted Price')
plt.ylabel('Actual Price')
plt.title('Actual vs Predicted')

plt.subplot(1, 2, 2)
plt.hist(residuals, bins=30, color='skyblue', edgecolor='black')
plt.axvline(0, color='red', linestyle='dashed', linewidth=2)
plt.xlabel('Residual (Error)')
plt.title('Error Distribution (Residuals)')

plt.tight_layout()
plt.show()

In [ ]:
from pytorch_forecasting import TimeSeriesDataSet

# 1. 설정값 정의
max_prediction_length = 30  # 30일 예측
max_encoder_length = 180    # 180일(6개월) 과거 데이터 참고
training_cutoff = tft_df["time_idx"].max() - max_prediction_length

# 2. 데이터셋 정의
training = TimeSeriesDataSet(
    tft_df[lambda x: x.time_idx <= training_cutoff],
    time_idx="time_idx",
    target="Copper_Price_Global",
    group_ids=["group"],
    min_encoder_length=max_encoder_length // 2,
    max_encoder_length=max_encoder_length,
    min_prediction_length=1,
    max_prediction_length=max_prediction_length,
    static_categoricals=["group"],
    time_varying_known_categoricals=["month"],
    time_varying_known_reals=["time_idx"],
    time_varying_unknown_reals=[
        "Copper_Price_Global",
        "PPI_Primary_Copper",
        "PPI_Secondary_Copper",
        "Housing_Starts",
        "PPI_Copper_Wire_Cable",
        "NASDAQ_Copper_Index"
    ],
    target_normalizer=GroupNormalizer(
        groups=["group"], transformation="softplus"
    ),  # 가격 데이터이므로 양수 보장하는 softplus 사용
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
)

# 3. 검증 데이터셋 생성
validation = TimeSeriesDataSet.from_dataset(training, tft_df, predict=True, stop_randomization=True)

# 4. 데이터로더 생성 (Dataloader)
batch_size = 64
train_dataloader = training.to_dataloader(train=True, batch_size=batch_size, num_workers=0)
val_dataloader = validation.to_dataloader(train=False, batch_size=batch_size * 10, num_workers=0)

print(f"학습용 샘플 수: {len(training)}")
print(f"검증용 샘플 수: {len(validation)}")

In [ ]:
import requests
import pandas as pd
import time

def fetch_un_comtrade_data_safely(hs_codes, start_year=2000, end_year=2023):
    """
    UN Comtrade API 제약(시간당 100회)을 준수하며 데이터를 호출합니다.
    """
    # 공용 API 엔드포인트 (인증키가 없을 경우 호출량에 제한이 더 엄격할 수 있음)
    base_url = "https://comtradeapi.un.org/public/v1/get/C/A/HS"
    all_data = []

    print(f"{start_year}년부터 {end_year}년까지 HS {hs_codes} 데이터 수집 시작...")

    for year in range(start_year, end_year + 1):
        for hs in hs_codes:
            # 전세계(World, reporterCode=0) 대상 수출입 데이터 요청
            params = {
                'period': str(year),
                'reporterCode': '0',
                'cmdCode': hs,
                'flowCode': 'M,X',
                'partnerCode': '0'
            }
            try:
                response = requests.get(base_url, params=params)
                if response.status_code == 200:
                    data = response.json().get('data', [])
                    if data:
                        all_data.extend(data)
                        print(f"  [성공] {year}년 HS {hs} 데이터 확보 ({len(data)}건)")
                    else:
                        print(f"  [확인] {year}년 HS {hs} 데이터가 해당 엔드포인트에 없습니다.")
                elif response.status_code == 429:
                    print("  [경고] 호출 제한 발생. 1분간 대기 후 재시도합니다.")
                    time.sleep(60)
            except Exception as e:
                print(f"  [에러] {year}년 HS {hs}: {e}")

            # 시간당 100회 제한을 고려한 지연 시간 (초)
            time.sleep(10)

    return pd.DataFrame(all_data)

# 1. UN Comtrade 실행 (부하를 줄이기 위해 우선 최근 5년치 테스트)
hs_codes = ['7403', '7401']
df_un_comtrade = fetch_un_comtrade_data_safely(hs_codes, start_year=2019)

# 2. USGS 기반 국가별/연도별 구리 생산량 데이터프레임 (2000~2023 요약)
usgs_data = [
    {'Year': 2000, 'Chile': 4602, 'Peru': 554, 'China': 593, 'World_Total': 13200, 'Reserves': 340000},
    {'Year': 2005, 'Chile': 5320, 'Peru': 1010, 'China': 750, 'World_Total': 15000, 'Reserves': 470000},
    {'Year': 2010, 'Chile': 5419, 'Peru': 1247, 'China': 1156, 'World_Total': 16100, 'Reserves': 630000},
    {'Year': 2015, 'Chile': 5760, 'Peru': 1700, 'China': 1700, 'World_Total': 19100, 'Reserves': 720000},
    {'Year': 2020, 'Chile': 5730, 'Peru': 2150, 'China': 1720, 'World_Total': 20600, 'Reserves': 870000},
    {'Year': 2021, 'Chile': 5620, 'Peru': 2210, 'China': 1910, 'World_Total': 21200, 'Reserves': 880000},
    {'Year': 2022, 'Chile': 5330, 'Peru': 2440, 'China': 1940, 'World_Total': 21900, 'Reserves': 890000},
    {'Year': 2023, 'Chile': 5000, 'Peru': 2600, 'China': 1700, 'World_Total': 22000, 'Reserves': 1000000}
]
df_usgs_global = pd.DataFrame(usgs_data)

print("\n[USGS 국가별 글로벌 구리 생산 통계 (kt)]")
display(df_usgs_global)

if not df_un_comtrade.empty:
    print("\n[UN Comtrade 데이터프레임 요약]")
    display(df_un_comtrade.head())
else:
    print("\nUN Comtrade: API 호출을 완료했으나 반환된 데이터가 없습니다. (URL 또는 필터 확인 필요)")

### D3.js 기반 국가별 생산량 인터랙티브 대시보드
아래 셀을 실행하면 연도별 슬라이더를 통해 국가별 구리 생산량 변화를 지도로 확인할 수 있습니다.

### 지도 시각화에 사용된 USGS 원본 데이터 확인
칠레(CHL), 페루(PER), 중국(CHN)의 생산량 데이터를 바탕으로 지도가 구성됩니다.

In [ ]:
import pandas as pd

# 지도에 사용된 핵심 데이터 다시 보기
display(df_usgs_global[['Year', 'Chile_Production_kt', 'Peru_Production_kt', 'China_Production_kt']])

In [ ]:
# @title 📊 Global Copper Production Dashboard {display-mode: "form"}
import json
import pandas as pd
from IPython.display import HTML
from google.colab import output

# JS 에러 리포팅 함수
def _report_js_error(message):
    print(f"JavaScript Error: {message}")

output.register_callback('report_js_error', _report_js_error)

# 데이터 준비
usgs_json = df_usgs_global.to_json(orient='records')

html_template = """
<!DOCTYPE html>
<html>
<head>
    <script src="https://d3js.org/d3.v7.min.js"></script>
    <style>
        body { margin: 0; font-family: 'Segoe UI', sans-serif; background: #f4f6f8; color: #333; }
        .dashboard-container { display: flex; flex-direction: column; height: 100vh; padding: 20px; box-sizing: border-box; }
        .header { background: white; padding: 15px 25px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.05); margin-bottom: 20px; display: flex; justify-content: space-between; align-items: center; }
        .main-content { display: flex; gap: 20px; flex-grow: 1; min-height: 0; }
        .viz-card { background: white; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.05); flex: 1; display: flex; flex-direction: column; padding: 20px; position: relative; }
        .controls { display: flex; align-items: center; gap: 15px; }
        #map-canvas { flex-grow: 1; min-height: 0; }
        .country { stroke: #fff; stroke-width: 0.5px; transition: fill 0.3s; cursor: pointer; }
        .tooltip { position: absolute; padding: 10px; background: rgba(0,0,0,0.85); color: white; border-radius: 5px; font-size: 13px; pointer-events: none; display: none; z-index: 1000; }
        input[type=range] { width: 150px; cursor: pointer; }
        .legend { position: absolute; bottom: 20px; right: 20px; background: rgba(255,255,255,0.8); padding: 10px; border-radius: 4px; font-size: 11px; }
    </style>
</head>
<body>
    <div class="dashboard-container">
        <div class="header">
            <h2 style="margin:0">World Copper Production Explorer</h2>
            <div class="controls">
                <span>Year: <strong id="yearDisplay" style="color:#d32f2f">2000</strong></span>
                <input type="range" id="yearSlider" min="0" value="0">
            </div>
        </div>
        <div class="main-content">
            <div class="viz-card">
                <div id="map-canvas"></div>
                <div id="tooltip" class="tooltip"></div>
                <div class="legend">
                    <div style="margin-bottom:5px"><strong>Production (kt)</strong></div>
                    <div id="legend-gradient" style="height:10px; width:100px; background: linear-gradient(to right, #fff7bc, #d95f0e);"></div>
                    <div style="display:flex; justify-content:between"><span>0</span><span style="margin-left:auto">6000+</span></div>
                </div>
            </div>
        </div>
    </div>

    <script>
    window.onerror = function(message) {
        google.colab.kernel.invokeFunction('report_js_error', [message], {});
    };

    const rawData = DATA_JSON_PLACEHOLDER;
    const width = 900;
    const height = 500;

    // 1. D3를 사용한 캔버스 생성 및 지리 투영 설정
    const svg = d3.select("#map-canvas").append("svg")
        .attr("viewBox", `0 0 ${width} ${height}`)
        .attr("width", "100%")
        .attr("height", "100%");

    const projection = d3.geoNaturalEarth1().scale(170).translate([width / 2, height / 1.6]);
    const path = d3.geoPath().projection(projection);

    // 2. D3의 스케일 함수를 이용한 색상 맵핑 정의
    const colorScale = d3.scaleSequential(d3.interpolateYlOrBr).domain([0, 6000]);
    const tooltip = d3.select("#tooltip");

    let worldData;

    // 3. 지리 데이터(GeoJSON) 로드
    d3.json("https://raw.githubusercontent.com/holtzy/D3-graph-gallery/master/DATA/world.geojson").then(geojson => {
        worldData = geojson;
        initSlider();
        updateMap(0);
    });

    function initSlider() {
        const slider = document.getElementById('yearSlider');
        slider.max = rawData.length - 1;
        slider.oninput = function() { updateMap(this.value); };
    }

    function updateMap(index) {
        const currentData = rawData[index];
        document.getElementById('yearDisplay').innerText = currentData.Year;

        const prodMap = {
            "Chile": currentData.Chile_Production_kt,
            "Peru": currentData.Peru_Production_kt,
            "China": currentData.China_Production_kt
        };

        // 4. D3 데이터 바인딩을 통한 지도 업데이트 로직
        svg.selectAll("path")
            .data(worldData.features)
            .join("path")
            .attr("d", path)
            .attr("class", "country")
            .style("fill", d => {
                const val = prodMap[d.properties.name];
                return val ? colorScale(val) : "#e0e0e0";
            })
            .on("mouseover", function(event, d) {
                const val = prodMap[d.properties.name];
                if (val) {
                    d3.select(this).style("opacity", 0.7);
                    tooltip.style("display", "block")
                        .html(`<strong>${d.properties.name}</strong><br>Production: ${val.toLocaleString()} kt`);
                }
            })
            .on("mousemove", function(event) {
                tooltip.style("left", (event.offsetX + 15) + "px")
                       .style("top", (event.offsetY - 15) + "px");
            })
            .on("mouseleave", function() {
                d3.select(this).style("opacity", 1);
                tooltip.style("display", "none");
            });
    }
    </script>
</body>
</html>
""".replace('DATA_JSON_PLACEHOLDER', usgs_json)

display(HTML(html_template))